<a href="https://colab.research.google.com/github/polreig/StartUp_DecoAI/blob/main/Notebook_final_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalación de las herramientas

In [ ]:
!pip install -q -U google-genai diffusers transformers accelerate opencv-python

Librerias y modelos

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
import requests
from io import BytesIO
from google import genai
from google.colab import userdata
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel

print("1. Cargando credenciales de Gemini...")
GOOGLE_API_KEY = userdata.get('clave_API_gemini')
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

print("2. Cargando ControlNet y Stable Diffusion a la GPU...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

print("¡Sistemas listos!")

El Cerebro

In [ ]:
def transformar_habitacion(ruta_o_url, peticion_usuario):
    print("📥 Cargando imagen original...")

    if ruta_o_url.startswith('http'):
        response = requests.get(ruta_o_url)
        imagen_original = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        imagen_original = Image.open(ruta_o_url).convert("RGB")

    imagen_original = imagen_original.resize((512, 512))

    # FASE 1: ANÁLISIS CON GEMINI (Motor INTACTO)
    print("🧠 Analizando la habitación con Gemini...")
    prompt_gemini = f"""
    Eres un diseñador de interiores. El cliente dice: '{peticion_usuario}'.
    Analiza la foto y responde estrictamente con este formato:

    ANÁLISIS PARA EL CLIENTE:
    - Estilo Actual: [Tu análisis]
    - Recomendación: [Tu recomendación basada en lo que pide]
    - Paleta de Colores: [Colores]

    LISTA_DE_COMPRA:
    [Escribe una lista EXHAUSTIVA de TODOS los muebles, iluminación, textiles y objetos decorativos principales que componen esta habitación (mínimo entre 8 y 15 productos). Sigue ESTE FORMATO EXACTO por línea:]
    - Nombre del producto | Tienda recomendada (Elige una: IKEA, Leroy Merlin, Zara Home, Amazon) | Precio estimado en euros

    PROMPT_IMAGEN:
    [Escribe aquí UNA SOLA FRASE EN INGLÉS, separada por comas, describiendo la habitación recomendada.
    Ejemplo: "A cozy rustic living room, wooden furniture, warm lighting, highly detailed, 8k resolution, photorealistic, interior design"]
    """

    respuesta_gemini = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[imagen_original, prompt_gemini]
    )

    texto_completo = respuesta_gemini.text

    partes_prompt = texto_completo.split("PROMPT_IMAGEN:")
    texto_previo = partes_prompt[0]
    prompt_sd = partes_prompt[1].strip() if len(partes_prompt) > 1 else "modern interior design, photorealistic, 8k"

    partes_lista = texto_previo.split("LISTA_DE_COMPRA:")
    analisis_cliente = partes_lista[0].strip()
    texto_lista = partes_lista[1].strip() if len(partes_lista) > 1 else ""

    # --- NUEVO DISEÑO VISUAL Y OMNI-TIENDA ---
    html_links = """
    <div style='font-family: "Segoe UI", sans-serif; background: #ffffff; padding: 30px; border-radius: 12px; box-shadow: 0 8px 24px rgba(0,0,0,0.08); border: 1px solid #eef0f2; max-width: 1000px; margin: 20px auto;'>
        <h2 style='color: #1a1a1a; margin-top: 0; margin-bottom: 5px; font-size: 24px;'>🛍️ Catálogo de Opciones</h2>
        <p style='color: #666; font-size: 14px; margin-bottom: 20px;'>Compara precios en diferentes tiendas para los elementos clave de tu nuevo diseño.</p>
        <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 15px;'>
    """

    for linea in texto_lista.split('\n'):
        if '|' in linea and linea.strip().startswith('-'):
            datos = linea.replace('-', '', 1).strip().split('|')
            if len(datos) >= 3:
                nombre = datos[0].strip()
                tienda = datos[1].strip()
                precio = datos[2].strip()

                # Generador de enlaces para TODAS las tiendas
                query = urllib.parse.quote(nombre)
                l_ikea = f"https://www.ikea.com/es/es/search/?q={query}"
                l_amazon = f"https://www.amazon.es/s?k={query}"
                l_leroy = f"https://www.leroymerlin.es/buscar?q={query}"
                l_zara = f"https://www.zarahome.com/es/search.html?keyword={query}"

                html_links += f"""
                <div style='background: #f8f9fa; border: 1px solid #e9ecef; border-radius: 10px; padding: 16px;'>
                    <div style='display: flex; justify-content: space-between; align-items: start; margin-bottom: 12px;'>
                        <h4 style='margin: 0; color: #2d3436; font-size: 15px; font-weight: 600;'>{nombre.title()}</h4>
                        <span style='background: #e8f5e9; color: #2e7d32; padding: 4px 8px; border-radius: 6px; font-size: 13px; font-weight: 600; white-space: nowrap; margin-left: 10px;'>{precio}</span>
                    </div>
                    <p style='margin: 0 0 12px 0; font-size: 12px; color: #636e72;'>✨ Sugerencia IA: <b>{tienda}</b></p>
                    <p style='margin: 0 0 8px 0; font-size: 11px; color: #b2bec3; text-transform: uppercase; letter-spacing: 0.5px;'>Comparar precios en:</p>
                    <div style='display: flex; flex-wrap: wrap; gap: 6px;'>
                        <a href='{l_ikea}' target='_blank' style='background: #0058a3; color: white; padding: 6px 12px; border-radius: 4px; font-size: 11px; text-decoration: none; font-weight: 600;'>IKEA</a>
                        <a href='{l_amazon}' target='_blank' style='background: #FF9900; color: #111; padding: 6px 12px; border-radius: 4px; font-size: 11px; text-decoration: none; font-weight: 600;'>Amazon</a>
                        <a href='{l_leroy}' target='_blank' style='background: #73c322; color: white; padding: 6px 12px; border-radius: 4px; font-size: 11px; text-decoration: none; font-weight: 600;'>Leroy Merlin</a>
                        <a href='{l_zara}' target='_blank' style='background: #111; color: white; padding: 6px 12px; border-radius: 4px; font-size: 11px; text-decoration: none; font-weight: 600;'>Zara Home</a>
                    </div>
                </div>
                """
    html_links += "</div></div>"

    # FASE 2 Y 3: ESTRUCTURA Y GENERACIÓN (Motor INTACTO)
    print("📐 Extrayendo plano de la habitación (Canny Edge)...")
    imagen_cv = np.array(imagen_original)
    bordes = cv2.Canny(imagen_cv, 100, 200)
    imagen_bordes = Image.fromarray(np.stack([bordes, bordes, bordes], axis=2))

    print("🎨 Dibujando 3 opciones del nuevo diseño...")
    prompt_negativo = "low quality, bad anatomy, worst quality, cartoon, illustration, distorted, messy"

    imagenes_generadas = pipe(
        prompt_sd,
        negative_prompt=prompt_negativo,
        image=imagen_bordes,
        num_inference_steps=25,
        num_images_per_prompt=3
    ).images

    return analisis_cliente, html_links, prompt_sd, imagen_original, imagen_bordes, imagenes_generadas

def obtener_3_estilos(ruta_imagen, peticion):
    # Cargamos la imagen rápidamente para que el Director la vea
    if ruta_imagen.startswith('http'):
        response = requests.get(ruta_imagen)
        img = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        img = Image.open(ruta_imagen).convert("RGB")
        
    print("👔 Director de Agencia: Analizando la foto para proponer 3 vías de diseño...")
    
    prompt = f"""
    Eres el Director de una Agencia de Diseño de Interiores. El cliente ha subido una foto de su habitación y su petición es: '{peticion}'.
    
    REGLAS ESTRICTAS:
    1. Si el cliente ha pedido un estilo concreto (ej: "quiero estilo nórdico"), el Estilo 1 será su petición mejorada, y los Estilos 2 y 3 serán alternativas tuyas que encajen bien con la arquitectura de la foto.
    2. Si el cliente dice "no lo sé", "sorpréndeme", o no indica nada claro, propón tú los 3 mejores estilos distintos para esa estancia.
    
    Devuelve EXACTAMENTE 3 líneas de texto. Nada de introducciones, ni viñetas, ni texto extra. Solo el nombre y una breve descripción del estilo en cada línea.
    
    EJEMPLO DE RESPUESTA:
    Estilo Industrial con toques de madera oscura y metal
    Estilo Nórdico Minimalista con tonos blancos y mucha luz natural
    Estilo Bohemio Cálido con plantas y alfombras étnicas
    """
    
    respuesta = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[img, prompt]
    )
    
    # Limpiamos la respuesta para asegurarnos de que tenemos una lista de 3 textos
    estilos = [linea.strip() for linea in respuesta.text.strip().split('\n') if linea.strip() and len(linea) > 5]
    
    # Seguro antierrores por si la IA se confunde
    if len(estilos) < 3:
        estilos.extend(["Estilo Moderno Elegante", "Estilo Rústico Acogedor", "Estilo Minimalista Contemporáneo"])
        
    return estilos[:3]

Prueba

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import urllib.parse

# Tus Datos (¡Prueba a cambiar la petición!)
mi_foto = "cuarto2.jpg.jpeg"

# CASO A: El cliente no lo sabe
mi_peticion = "No lo sé, no tengo ni idea de decoración. Sorpréndeme con lo que creas que le queda mejor."

# CASO B: El cliente sí lo sabe (Descomenta esta línea para probarlo)
# mi_peticion = "Quiero un estilo industrial con ladrillo y metal."

try:
    # 1. El Director decide los 3 estilos
    tres_estilos = obtener_3_estilos(mi_foto, mi_peticion)
    
    print("\n" + "="*60)
    print("🎯 EL DIRECTOR HA DECIDIDO ESTAS 3 PROPUESTAS PARA TI:")
    for i, est in enumerate(tres_estilos):
        print(f"   Vía {i+1}: {est}")
    print("="*60 + "\n")
    
    # 2. El Motor hace su magia 3 veces (Una por cada estilo)
    for i, estilo_actual in enumerate(tres_estilos):
        print(f"\n🚀 --- INICIANDO RENDERIZADO DE PROPUESTA {i+1} ---")
        
        # LLAMAMOS A TU MOTOR INTACTO pasándole el estilo exacto
        analisis, lista_html, prompt_usado, img_orig, img_bordes, opciones_generadas = transformar_habitacion(mi_foto, estilo_actual)
        
        # 3. Presentación espectacular por cada bloque
        display(HTML(f"<h1 style='color: #2c3e50; text-align: center; border-bottom: 3px solid #3498db; padding-bottom: 10px; margin-top: 40px;'>🌟 PROPUESTA {i+1}: {estilo_actual.upper()} 🌟</h1>"))
        
        print(analisis)
        display(HTML(lista_html))
        
        # Pintamos las 4 fotos (Original + 3 Opciones) de esta propuesta
        fig, axes = plt.subplots(1, 4, figsize=(24, 6))
        fig.patch.set_facecolor('#f8f9fa')
        
        # Original
        axes[0].imshow(img_orig)
        axes[0].set_title("FOTO ORIGINAL", fontweight="bold", color="gray", pad=15)
        axes[0].axis("off")
        
        # Las 3 opciones de la IA
        colores = ['#2980b9', '#27ae60', '#8e44ad']
        for j, img in enumerate(opciones_generadas):
            axes[j+1].imshow(img)
            axes[j+1].set_title(f"Variante {j+1}", fontweight="bold", color=colores[j], pad=15, fontsize=14)
            axes[j+1].axis("off")
            for spine in axes[j+1].spines.values():
                spine.set_visible(True)
                spine.set_color('#dddddd')
                spine.set_linewidth(2)
                
        plt.tight_layout()
        plt.show()
        
except Exception as e:
    print(f"❌ Error en la ejecución: {e}")